# f6_m03a_fairness.ipynb

## Qué hace
Analiza la equidad algorítmica del modelo CatBoost__balanced usando fairlearn.
Evalúa si el modelo predice el abandono de forma homogénea entre grupos definidos
por sexo, rama de conocimiento, vía de acceso y situación de beca.
Métricas: Demographic Parity, Equal Opportunity, Equalized Odds, Disparate Impact.

## Requisitos
- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/models/CatBoost__balanced.pkl`
- `data/03_features/df_exp_target_eda.parquet` (para titulacion y rama)
- `data/05_modelado/X_test.parquet` (índices originales para join)
- Paquete: fairlearn

## Genera
- `results/fase6/fairness_metricas.parquet` — métricas por grupo y variable sensible
- `results/fase6/fairness_sexo.png`
- `results/fase6/fairness_rama.png`
- `results/fase6/fairness_via_acceso.png`
- `results/fase6/fairness_tuvo_beca.png`
- `docs/html/fase6/m03a_fairness.html`

## Flujo
Cargar datos → calcular predicciones → join grupos sensibles →
métricas fairlearn por grupo → gráficos → HTML

## Siguiente
`f6_m03b_errores_fpfn.ipynb` — análisis de falsos positivos y falsos negativos

In [1]:
# 1. CONFIGURACIÓN DE RUTAS
# ROOT detectado subiendo niveles hasta encontrar src/ — nunca hardcodeado
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_DATA     = ROOT / 'data' / '05_modelado'
DIR_FEATURES = ROOT / 'data' / '03_features'
DIR_MODELS   = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS  = ROOT / 'results' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)

print(f'ROOT:        {ROOT}')
print(f'DIR_RESULTS: {DIR_RESULTS}')

ROOT:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
DIR_RESULTS: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\results\fase6


In [2]:
# 2. IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import joblib
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference,
    equal_opportunity_difference,
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from src.html.render import render_pagina_desde_fichero

matplotlib.rcParams['figure.dpi'] = 120

print('Imports OK.')

Imports OK.


In [3]:
# 3. CARGAR DATOS Y MODELO
# Cargamos X_test (con índices originales) para el join con df_ref.
# Las probabilidades y predicciones se calculan con predict_proba/predict.
# Los modelos de Fase 5 son sklearn.Pipeline — no extraer named_steps aquí
# porque predict y predict_proba funcionan directamente sobre el Pipeline.

X_test       = pd.read_parquet(DIR_DATA / 'X_test.parquet')
X_test_prep  = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
y_test       = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
pipeline_cat = joblib.load(DIR_MODELS / 'CatBoost__balanced.pkl')

# Calcular predicciones y probabilidades
y_pred = pipeline_cat.predict(X_test_prep)
y_prob = pipeline_cat.predict_proba(X_test_prep)[:, 1]
y_true = y_test.values.ravel()

print(f'X_test_prep: {X_test_prep.shape}')
print(f'Tasa abandono real:     {y_true.mean():.2%}')
print(f'Tasa abandono predicha: {y_pred.mean():.2%}')

X_test_prep: (6725, 19)
Tasa abandono real:     29.25%
Tasa abandono predicha: 31.87%


In [4]:
# 4. JOIN VARIABLES SENSIBLES
# Todas las variables sensibles se recuperan de df_exp_target_eda
# para tener las etiquetas originales, no los códigos numéricos
# que genera el preprocesado de Fase 5.

df_ref  = pd.read_parquet(DIR_FEATURES / 'df_exp_target_eda.parquet')
df_meta = df_ref[['sexo', 'rama', 'via_acceso', 'tuvo_beca']].iloc[X_test.index].copy()
df_meta.index = X_test.index

df_grupos = pd.DataFrame({
    'sexo':       df_meta['sexo'].values,
    'rama':       df_meta['rama'].values,
    'via_acceso': df_meta['via_acceso'].values,
    'tuvo_beca':  df_meta['tuvo_beca'].map({0: 'Sin beca', 1: 'Con beca'}).fillna('Sin beca').values,
    'y_true':     y_true,
    'y_pred':     y_pred,
    'y_prob':     y_prob,
})

print('Grupos sensibles:')
for col in ['sexo', 'rama', 'via_acceso', 'tuvo_beca']:
    vals = df_grupos[col].value_counts()
    print(f'  {col}: {vals.to_dict()}')

Grupos sensibles:
  sexo: {'Mujer': 3705, 'Hombre': 3020}
  rama: {'SO': 3761, 'TE': 1532, 'SA': 686, 'HU': 569, 'EX': 177}
  via_acceso: {'Pruebas acceso Bachiller Logse': 4933, 'Ciclo Formativo de Grado sup. o equivalente': 804, 'Adaptación a Grado': 300, 'Titulados Universitarios': 269, 'Pruebas acceso mayores 25 años': 128, 'Traslado': 109, 'Extranjeros CEE': 87, 'Extranjeros no CEE': 56, 'Cambio de plan': 16, 'Pruebas acceso mayores 40 años': 12, 'Pruebas acceso mayores 45 años': 8, 'Deportistas de élite': 2, 'Minusválidos': 1}
  tuvo_beca: {'Con beca': 4869, 'Sin beca': 1856}


In [5]:
# 5. CALCULAR MÉTRICAS DE EQUIDAD POR GRUPO
# MetricFrame calcula métricas desagregadas por grupo sensible.
# Métricas por grupo: accuracy, precision, recall (= tasa detección abandono), F1.
# Métricas globales de equidad:
#   - Demographic Parity Difference: diferencia en tasa de predicción positiva entre grupos
#   - Equal Opportunity Difference: diferencia en recall (TPR) entre grupos
#   - Disparate Impact Ratio: ratio de tasa positiva entre grupo peor/mejor (ideal=1.0)

metricas_fn = {
    'accuracy':  accuracy_score,
    'precision': lambda y_t, y_p: precision_score(y_t, y_p, zero_division=0),
    'recall':    lambda y_t, y_p: recall_score(y_t, y_p, zero_division=0),
    'f1':        lambda y_t, y_p: f1_score(y_t, y_p, zero_division=0),
}

variables_sensibles = ['sexo', 'rama', 'via_acceso', 'tuvo_beca']
resultados_equidad = []

for var in variables_sensibles:
    mf = MetricFrame(
        metrics=metricas_fn,
        y_true=df_grupos['y_true'],
        y_pred=df_grupos['y_pred'],
        sensitive_features=df_grupos[var]
    )

    dpd = demographic_parity_difference(
        df_grupos['y_true'], df_grupos['y_pred'],
        sensitive_features=df_grupos[var]
    )
    dpr = demographic_parity_ratio(
        df_grupos['y_true'], df_grupos['y_pred'],
        sensitive_features=df_grupos[var]
    )
    eod = equal_opportunity_difference(
        df_grupos['y_true'], df_grupos['y_pred'],
        sensitive_features=df_grupos[var]
    )

    resultados_equidad.append({
        'variable':     var,
        'dp_diff':      round(dpd, 4),
        'dp_ratio':     round(dpr, 4),
        'eq_opp_diff':  round(eod, 4),
        'metric_frame': mf,
    })

    print(f'\n--- {var.upper()} ---')
    print(f'  Demographic Parity Difference: {dpd:.4f}  (ideal=0)')
    print(f'  Demographic Parity Ratio:      {dpr:.4f}  (ideal=1)')
    print(f'  Equal Opportunity Difference:  {eod:.4f}  (ideal=0)')
    print(mf.by_group.round(3).to_string())


--- SEXO ---
  Demographic Parity Difference: 0.1663  (ideal=0)
  Demographic Parity Ratio:      0.5947  (ideal=1)
  Equal Opportunity Difference:  0.0538  (ideal=0)
        accuracy  precision  recall     f1
sexo                                      
Hombre     0.835      0.747   0.833  0.788
Mujer      0.885      0.737   0.779  0.757



--- RAMA ---
  Demographic Parity Difference: 0.2560  (ideal=0)
  Demographic Parity Ratio:      0.4039  (ideal=1)
  Equal Opportunity Difference:  0.0969  (ideal=0)
      accuracy  precision  recall     f1
rama                                    
EX       0.927      0.931   0.857  0.893
HU       0.868      0.735   0.760  0.747
SA       0.927      0.731   0.829  0.777
SO       0.872      0.755   0.815  0.784
TE       0.800      0.708   0.802  0.752



--- VIA_ACCESO ---
  Demographic Parity Difference: 0.7500  (ideal=0)
  Demographic Parity Ratio:      0.0000  (ideal=1)
  Equal Opportunity Difference:  1.0000  (ideal=0)
                                             accuracy  precision  recall     f1
via_acceso                                                                     
Adaptación a Grado                              0.793      0.808   0.787  0.797
Cambio de plan                                  0.438      0.300   0.600  0.400
Ciclo Formativo de Grado sup. o equivalente     0.862      0.813   0.832  0.822
Deportistas de élite                            0.500      0.000   0.000  0.000
Extranjeros CEE                                 0.747      0.658   0.735  0.694
Extranjeros no CEE                              0.536      0.364   0.706  0.480
Minusválidos                                    1.000      0.000   0.000  0.000
Pruebas acceso Bachiller Logse                  0.875      0.737   0.813  0.773
Pruebas acceso mayores 25 a


--- TUVO_BECA ---
  Demographic Parity Difference: 0.2810  (ideal=0)
  Demographic Parity Ratio:      0.4618  (ideal=1)
  Equal Opportunity Difference:  0.1518  (ideal=0)
           accuracy  precision  recall     f1
tuvo_beca                                    
Con beca      0.864      0.670   0.740  0.703
Sin beca      0.859      0.831   0.891  0.860


In [6]:
# 6. GUARDAR MÉTRICAS
# Guardamos tabla resumen para el informe final y Streamlit Fase 7.

registros = []
for res in resultados_equidad:
    var = res['variable']
    mf  = res['metric_frame']
    for grupo, row in mf.by_group.iterrows():
        registros.append({
            'variable_sensible': var,
            'grupo':             str(grupo),
            'accuracy':          row['accuracy'],
            'precision':         row['precision'],
            'recall':            row['recall'],
            'f1':                row['f1'],
            'dp_diff':           res['dp_diff'],
            'dp_ratio':          res['dp_ratio'],
            'eq_opp_diff':       res['eq_opp_diff'],
        })

df_fairness = pd.DataFrame(registros)
df_fairness.to_parquet(DIR_RESULTS / 'fairness_metricas.parquet')
print(f'Métricas guardadas: {df_fairness.shape}')
print(df_fairness[['variable_sensible','grupo','recall','f1','dp_diff']].to_string())

Métricas guardadas: (22, 9)
   variable_sensible                                        grupo    recall        f1  dp_diff
0               sexo                                       Hombre  0.832734  0.787750   0.1663
1               sexo                                        Mujer  0.778947  0.757248   0.1663
2               rama                                           EX  0.857143  0.892562   0.2560
3               rama                                           HU  0.760274  0.747475   0.2560
4               rama                                           SA  0.828571  0.776786   0.2560
5               rama                                           SO  0.815299  0.784208   0.2560
6               rama                                           TE  0.802065  0.752220   0.2560
7         via_acceso                           Adaptación a Grado  0.787097  0.797386   0.7500
8         via_acceso                               Cambio de plan  0.600000  0.400000   0.7500
9         via_acceso  

In [7]:
# 7. GRÁFICOS — RECALL Y F1 POR GRUPO PARA CADA VARIABLE SENSIBLE
# Recall es la métrica más importante en detección de abandono:
# un recall bajo en un grupo significa que el modelo falla más en detectar
# el abandono real de ese grupo — consecuencia directa de inequidad.

rutas_fairness = {}
colores_barra = ['#3182ce', '#e53e3e', '#38a169', '#dd6b20',
                 '#805ad5', '#d69e2e', '#319795', '#e53e8e']

for res in resultados_equidad:
    var = res['variable']
    mf  = res['metric_frame']
    ruta = DIR_RESULTS / f'fairness_{var}.png'
    rutas_fairness[var] = ruta

    grupos  = mf.by_group.index.tolist()
    recalls = mf.by_group['recall'].tolist()
    f1s     = mf.by_group['f1'].tolist()
    x       = np.arange(len(grupos))
    ancho   = 0.35

    fig, ax = plt.subplots(figsize=(max(8, len(grupos) * 1.2), 5))
    bars1 = ax.bar(x - ancho/2, recalls, ancho, label='Recall', color='#3182ce', alpha=0.8)
    bars2 = ax.bar(x + ancho/2, f1s,     ancho, label='F1',     color='#e53e3e', alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels([str(g)[:20] for g in grupos], rotation=30, ha='right', fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.axhline(mf.overall['recall'], color='#3182ce', linestyle='--',
               linewidth=1, alpha=0.6, label=f'Recall global ({mf.overall["recall"]:.2f})')
    ax.axhline(mf.overall['f1'], color='#e53e3e', linestyle='--',
               linewidth=1, alpha=0.6, label=f'F1 global ({mf.overall["f1"]:.2f})')
    ax.set_ylabel('Métrica')
    ax.set_title(
        f'Equidad — {var} | DP_diff={res["dp_diff"]:.3f} | EqOpp_diff={res["eq_opp_diff"]:.3f}',
        fontsize=12, pad=10
    )
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(ruta, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'Gráfico guardado: {ruta.name}')

Gráfico guardado: fairness_sexo.png
Gráfico guardado: fairness_rama.png


Gráfico guardado: fairness_via_acceso.png


Gráfico guardado: fairness_tuvo_beca.png


In [8]:
# 8. GENERAR HTML
# render_pagina_desde_fichero — estándar del proyecto.

import base64

def img_b64(ruta: Path) -> str:
    if not ruta.exists():
        return ''
    with open(ruta, 'rb') as f:
        return base64.b64encode(f.read()).decode()

def bloque_imagen(b64: str, titulo: str, caption: str) -> str:
    if not b64:
        return f'<p style="color:#e53e3e">Imagen no disponible: {titulo}</p>'
    return (
        '<div style="margin:24px 0">'
        f'<h3 style="color:#2d3748; font-size:15px">{titulo}</h3>'
        f'<img src="data:image/png;base64,{b64}" style="max-width:100%; border-radius:6px; box-shadow:0 2px 8px rgba(0,0,0,.1)">'
        f'<p style="color:#718096; font-size:12px; margin-top:6px">{caption}</p>'
        '</div>'
    )

# Tabla resumen de métricas globales de equidad
filas_resumen = ''
for res in resultados_equidad:
    var    = res['variable']
    dpd    = res['dp_diff']
    dpr    = res['dp_ratio']
    eod    = res['eq_opp_diff']
    # Semáforo: verde si abs(dpd)<0.1, amarillo si <0.2, rojo si >=0.2
    if abs(dpd) < 0.1:
        color_dpd = '#38a169'; icono = '✅'
    elif abs(dpd) < 0.2:
        color_dpd = '#d69e2e'; icono = '⚠️'
    else:
        color_dpd = '#e53e3e'; icono = '❌'
    filas_resumen += (
        '<tr>'
        f'<td style="padding:8px 12px; font-weight:600">{var}</td>'
        f'<td style="padding:8px 12px; text-align:center; color:{color_dpd}">{icono} {dpd:.4f}</td>'
        f'<td style="padding:8px 12px; text-align:center">{dpr:.4f}</td>'
        f'<td style="padding:8px 12px; text-align:center">{eod:.4f}</td>'
        '</tr>'
    )

contenido = (
    '<h2 style="color:#2d3748">Fairness — Equidad Algorítmica</h2>'
    '<p style="color:#4a5568; font-size:14px; max-width:800px">'
    'Análisis de equidad del modelo CatBoost sobre cuatro variables sensibles: '
    'sexo, rama de conocimiento, vía de acceso y situación de beca. '
    'Se evalúa si el modelo predice el abandono con la misma precisión en todos los grupos '
    'o si existen disparidades sistemáticas que comprometan su uso equitativo.'
    '</p>'
    '<h3 style="color:#2d3748; margin-top:20px">Resumen de métricas de equidad</h3>'
    '<table style="width:100%; border-collapse:collapse; font-size:13px; margin-bottom:24px">'
    '<thead><tr style="background:#edf2f7">'
    '<th style="padding:8px 12px; text-align:left">Variable sensible</th>'
    '<th style="padding:8px 12px; text-align:center">DP Difference<br><small>(ideal=0)</small></th>'
    '<th style="padding:8px 12px; text-align:center">DP Ratio<br><small>(ideal=1)</small></th>'
    '<th style="padding:8px 12px; text-align:center">EqOpp Difference<br><small>(ideal=0)</small></th>'
    '</tr></thead>'
    f'<tbody>{filas_resumen}</tbody></table>'
    + bloque_imagen(img_b64(rutas_fairness['sexo']),
        'Equidad por sexo',
        'Recall y F1 por género. Líneas discontinuas = métricas globales. '
        'Diferencias grandes indican que el modelo detecta el abandono mejor en un género que en otro.')
    + bloque_imagen(img_b64(rutas_fairness['rama']),
        'Equidad por rama de conocimiento',
        'Recall y F1 por rama. Permite detectar si hay ramas donde el modelo falla sistemáticamente.')
    + bloque_imagen(img_b64(rutas_fairness['via_acceso']),
        'Equidad por vía de acceso',
        'Recall y F1 por vía de acceso (selectividad, FP, mayores 25, etc.).')
    + bloque_imagen(img_b64(rutas_fairness['tuvo_beca']),
        'Equidad por situación de beca',
        'Recall y F1 para alumnos con y sin beca. Crítico dado que n_anios_beca '
        'es la variable más importante del modelo — verificar que no hay sesgo sistemático.')
    + '<div style="margin-top:24px; padding:16px; background:#ebf8ff; '
    + 'border-left:4px solid #3182ce; border-radius:6px; font-size:13px; color:#2c5282">'
    + '<strong>Interpretación:</strong> DP Difference &lt; 0.1 indica equidad aceptable (✅). '
    + 'Entre 0.1 y 0.2 requiere atención (⚠️). Por encima de 0.2 indica disparidad '
    + 'significativa que debería abordarse antes de desplegar el modelo (❌).'
    + '</div>'
)

html_completo = render_pagina_desde_fichero('f6_m03a_fairness.ipynb', contenido)
ruta_html = ROOT / 'docs' / 'html' / 'fase6' / 'm03a_fairness.html'
ruta_html.parent.mkdir(parents=True, exist_ok=True)
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'HTML generado: {ruta_html}')

HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase6\m03a_fairness.html
